# Synthetic tRBS cases, part 1: generating a case

**This is part 1 of 2.** Part 1 builds cases. [Part 2](02_optimising_a_synthetic_case.ipynb)
optimises them and measures how close the optimiser got.

## In one minute

When we optimise a real tRBS case, we get an allocation and a score. What we
never get is the answer to the obvious question: **was that actually the best
allocation possible?** On a real case nobody knows the true optimum, so there is
nothing to compare against.

These two notebooks show the way around that. We *build* tRBS cases whose best
allocation we work out in advance, then feed them to the optimiser and measure
exactly how far off it was.

- **The problem:** on real cases, "did the optimiser find the best plan?" cannot
  be answered.
- **What this does:** generates realistic tRBS cases where the best plan is
  known beforehand.
- **What you get:** an exact error number (the *gap*) for any optimisation
  method, at any problem size.

The generated cases are ordinary tRBS cases. They are written as the same tables
a real case uses and run through the same pipeline, so the optimiser cannot tell
them apart from a real one.

## 1. Why this matters

The main contribution of this thesis is a **convexity characterisation** of tRBS
cases: a rule that says when the appreciation landscape is convex, and whether
that predicts if an optimiser finds the true best allocation.

A useful picture:

- **Convex** is a single mountain. Walk uphill from wherever you start and you
  always end up at the same summit, which is the highest point.
- **Non-convex** is a mountain range. Walk uphill and you reach the nearest
  peak, which is not necessarily the highest one. Where you started decides
  where you end up.

On real cases such as Beerwiser, Refugee and IZZ we cannot check which situation
we are in, because the true summit is unknown. Claiming "the optimiser found the
best plan" would be an act of faith. Generated cases replace that faith with a
measurement.

## 2. Setup

Two things to know before running this notebook.

Run it in the repository's virtual environment, the one where `vlinder` is
installed in editable mode, and not against a plain `pip install vlinder`. The
notebook needs both the installed `vlinder` package and the thesis modules
`case_factory` and `oracle`, which are not shipped with the released package.

The working directory must be `experiments/synthetic/`, which it is, since the
notebook lives there. That is what makes the plain `import case_factory` work
and puts generated cases in `experiments/synthetic/generated/`.

In [ ]:
import pandas as pd

import case_factory as cf

## 3. Which cases can you make?

Start here, because it saves guessing at parameter combinations. The table below
is the **standard suite**: one case per regime and per mechanism, ordered from
the easiest landscape to the hardest. It is generated from
`cf.standard_cases()`, so it is the real suite rather than a hand-typed copy
that could drift out of date.

Read the `landscape` column first. That is what actually changes the difficulty
of the optimisation problem; everything else is the knob that produces it.

In [ ]:
def describe(p):
    """Plain-language landscape plus the knobs that switch it on."""
    if p.regime == "convex":
        landscape = "one peak, at a corner" if p.appreciation == "linear" else "one peak, inside the region"
    elif p.regime == "smooth_nonconvex":
        landscape = "rounded peaks, no sharp edges"
    else:
        landscape = "flat plateaus with sharp creases"

    knobs = []
    for field, label in [
        ("n_stb1", "n_stb1"),
        ("n_bilinear", "n_bilinear"),
        ("n_saturation", "n_saturation"),
        ("scenario_dispersion", "scenario_dispersion"),
    ]:
        if getattr(p, field):
            knobs.append(f"{label}={getattr(p, field)}")
    if p.bracketing_factor < 1.0:
        knobs.append(f"bracketing_factor={p.bracketing_factor}")

    return {
        "name": p.name,
        "regime": p.regime,
        "appreciation": p.appreciation,
        "k": p.k,
        "landscape": landscape,
        "mechanisms": ", ".join(knobs) if knobs else "none (all neutral)",
    }


catalogue = pd.DataFrame([describe(p) for p in cf.standard_cases()])
catalogue

The first six rows are the easy end: a single peak, so any sensible optimiser
should find it and a miss means something is wrong with the method. The rows
below them are where the research question lives, because there the optimiser
can get stuck on the wrong peak.

Note that harder is not the same as bigger. `Synthetic_convex_k9` has nine
internal variables and is still a single mountain, while
`Synthetic_nonsmooth_clip_k3` has three and can trap a solver. Size and
difficulty are separate knobs, which is exactly why the study varies them
independently.

## 4. How a case is described

The generator writes a case as the 11 standard tRBS tables (`key_outputs`,
`decision_makers_options`, `dependencies`, `scenarios` and so on). tRBS reads
those tables from CSV, Excel or JSON; here we write CSV. Because the tables are
the real thing, the case flows through the normal
`build` to `evaluate` to `appreciate` to `optimize` pipeline untouched.

Alongside the tables the generator writes a `manifest.json` holding the settings
that produced the case, its convexity claim, and later the certified best
allocation computed by the **oracle** (part 2).

Three settings do most of the work:

- **`regime`**: the shape of the landscape, which is the `landscape` column
  above. `convex` (one mountain), `smooth_nonconvex` (several rounded peaks) or
  `nonsmooth` (peaks with sharp ridges and flat plateaus).
- **`k`**: the number of internal variable inputs (vlinder's name for the things
  we can spend the budget on). This is the size of the problem.
- **`seed`**: the starting value of the random number generator. Everything
  random in a case (the coefficients, the weights) is drawn from it, so the same
  seed always rebuilds exactly the same case, file for file. It is what makes
  the results reproducible by anyone.

Section 4.1 below lists every setting; the trickier ones get their own worked
explanation there.


### 4.1 Every setting, and what it does

| setting | default | what it does |
| --- | --- | --- |
| `name` | derived | folder name for the case; empty derives one from the settings (Section 5.1) |
| `k` | 3 | number of internal variable inputs, the size of the problem |
| `budget` | 100 | the budget $B$; feasible allocations satisfy $\sum_i x_i \le B$ |
| `n_key_outputs` | 3 | number of KPIs |
| `themes` | People, Planet, Profit | the tRBS themes the KPIs cycle through |
| `n_scenarios` | 3 | number of scenarios |
| `regime` | convex | shape of the landscape: `convex`, `smooth_nonconvex` or `nonsmooth` |
| `appreciation` | linear | `linear` (optimum in a corner) or `sinusoidal` (concave, interior optimum possible) |
| `seed` | 0 | drives every random draw; same seed, byte-identical case |
| `coef_low`, `coef_high` | 0.5, 1.5 | range of the input-to-KPI coefficients (see below) |
| `n_stb1` | 0 | KPIs flipped to the *decreasing* sinusoidal appreciation, which is convex and can carry a second peak (`smooth_nonconvex` only) |
| `n_bilinear` | 0 | KPIs given a product of two inputs (see below; `smooth_nonconvex` only) |
| `bracketing_factor` | 1.0 | below 1 shrinks the DMO grid and activates score clipping (see below; `nonsmooth` only) |
| `n_saturation` | 0 | inputs given a saturation cap $\min\{x/sp,\,1\}$, a kink without clipping (`nonsmooth` only) |
| `saturation_point` | 0.5 | where saturation kicks in, as a share of the budget |
| `scenario_dispersion` | 0.0 | strength of the per-KPI scenario factors (see below) |
| `scenario_mode` | independent | how those factors are drawn: `independent` or `coherent` (see below) |

The regime-specific settings are deliberately exclusive: a `convex` case must
keep all of them neutral, and each mechanism belongs to one regime, so the
factorial design stays clean and the analytic envelope stays exact.

**`coef_low` and `coef_high`.** Every KPI $j$ receives a contribution
$c_{ji}\,x_i$ from every input $i$, with each coefficient drawn uniformly from
$[\texttt{coef\_low}, \texttt{coef\_high}]$. Both ends positive means every
KPI rises monotonically in every input. The width of the range controls how much
the inputs differ in effectiveness, and with that how strongly the optimum
concentrates the budget on the best inputs.

**`n_bilinear`.** The first `n_bilinear` KPIs additionally receive one term
$q\,x_a x_b$ for a pair of inputs. The Hessian of such a product is indefinite,
so it bends the landscape in a way that can create a second peak; it is the
generator's smooth route to multi-modality. The pair and the strength $q$ are
drawn from the seed.

**`bracketing_factor`.** tRBS puts the 0-to-100 appreciation scale on the lowest
and highest KPI values it observes across the DMO grid. At the default 1.0 the
bracketing DMOs (Section 5.3) span the exact feasible range, so no score is ever
clipped. Below 1.0 every DMO row $v$ is pulled toward the equal spread,
$f\,v + (1-f)\,\text{equal}$, the observed range shrinks, and allocations
outside it are clipped at score 0 inside the feasible set. That clipping is the
mechanism under study: it kinks the landscape and creates plateaus and extra
local optima.

**`scenario_dispersion`, `scenario_mode`, and `Ext` versus `Fac`.** A tRBS
scenario is a set of values for the external variable inputs, so every case
carries at least one: `Ext 01`, an *additive* term on KPI 1 worth about 10% of
the budget, spread over the scenarios. It exists to make the scenarios real, but
it is deliberately harmless: it does not depend on the allocation, so it shifts
every allocation's score equally and never changes which allocation is best.
`Fac 01..n` appear only when `scenario_dispersion > 0` and are *multiplicative*,
one factor per KPI per scenario, so they reweight the KPIs across scenarios and
can genuinely move the optimum.

How the factors are drawn is the `scenario_mode`:

- `independent` (default): every factor is its own uniform draw from
  $1 \pm \text{dispersion}$. One KPI can improve in the very scenario where
  another worsens; scenarios are stress patterns rather than a story, which is
  why the synthetic scenarios are simply named `Scenario 01..0S`.
- `coherent`: scenarios are ordered worst to best and each KPI draws one
  *sensitivity* in $[0.2, 1.0]$. The factor is then
  $\text{Fac} = 1 + \text{sensitivity} \times \text{dispersion} \times \text{position}$,
  with positions evenly spaced on $[-1, +1]$. Everything moves down together in
  the worst scenario and up together in the best, but by different amounts, so
  stable KPIs weigh relatively more in bad times and the optimum can shift.
  Example with dispersion 0.5 and sensitivities $[0.9, 0.3, 0.7]$: the worst
  scenario multiplies the KPIs by $[0.55, 0.85, 0.65]$, the middle by
  $[1, 1, 1]$, the best by $[1.45, 1.15, 1.35]$. The factor is floored at
  $0.1$, so even sensitivity 1.0 with dispersion 1.0 flattens a KPI's
  contribution without deleting it from the objective.


## 5. Generate a simple case

We start at the top of the table: a convex case with three internal variables
and three KPIs.

- `k=3`: three internal variables, so three things to divide the budget over.
- `n_key_outputs=3`: three KPIs to score the result on.
- `seed=1`: fixes the random draws, so this exact case can be rebuilt later.

Everything else stays at its default: `regime='convex'`,
`appreciation='linear'`, and `budget=100.0`, meaning any allocation has to
satisfy sum(x) <= 100.

The parameters that would introduce non-convexity are all switched off, which
the convex regime requires: `n_stb1=0` (no KPIs flipped to smaller-is-better),
`n_bilinear=0` (no multiplicative couplings between variables),
`n_saturation=0` (no capped variables) and `bracketing_factor=1.0` (no
clipping). Section 7 turns these on one at a time.

In [ ]:
params = cf.SyntheticCaseParams(k=3, n_key_outputs=3, seed=1)
params

### 5.1 The name comes from the settings

Notice we did not pass a name. Naming every case by hand is busywork, and it
used to be worse than that: the default was one fixed string, so two
differently-configured cases written to the same folder would quietly overwrite
each other.

The name is now derived from the settings that actually distinguish one case
from another (regime, active mechanisms, `k` and `seed`), so two cases collide
only when they really are the same case. Pass `name=` when you want something
specific, for example when a case is a fixture in a paper or a test.

In [ ]:
print("derived: ", cf.SyntheticCaseParams(k=6, regime="nonsmooth", bracketing_factor=0.7, seed=5).name)
print("derived: ", cf.SyntheticCaseParams(k=3, appreciation="sinusoidal", seed=1).name)
print("override:", cf.SyntheticCaseParams(k=3, seed=1, name="My_own_case").name)

### 5.2 Write it to disk

`.write()` produces the 11 tRBS tables under `<root>/<name>/csv/` plus the
`manifest.json`. Running it twice with the same parameters is safe: it writes
exactly the same bytes.

In [ ]:
root = cf.SyntheticCaseFactory(params).write()
print(f"Case written to: {root / params.name}")

### 5.3 Look at what came out

These are ordinary tRBS tables. Below are the decision makers options, which set
the range each internal variable can move in.

The DMO rows are the **bracketing grid**, and each kind has one job:

- **`Corner 01..k`** put the whole budget on one input; together with
  **`Lowest spend`** (all zeros; before the review this row was called
  `Zero spend`, renamed because a `bracketing_factor` below 1 pulls it toward
  the equal spread, so it is the lowest spend the case observes rather than
  zero) and **`Equal spread`** they realise the extreme KPI values of an affine
  case, so tRBS's automatic score boundaries equal the exact feasible range.
- **`Blend j`** exists only when `n_bilinear > 0`. A bilinear KPI can peak on an
  *edge* of the budget face rather than in a corner, so its maximum would be
  missed by corners alone; the Blend row is the closed-form maximiser of KPI
  $j$ along the edge between its bilinear pair.
- **`Envelope j`** exists only when saturation is active. It fills the budget
  greedily in descending coefficient order for KPI $j$, giving each input at
  most its saturation cap, so the observed maximum of every KPI is its true
  feasible maximum even when inputs saturate.

Two rows can coincide: when the strongest input for KPI $j$ is not saturated,
`Envelope j` receives the whole budget on that input and equals its Corner, and
two KPIs that share their strongest input share their Envelope. Duplicates are
left in deliberately. The boundaries are a minimum and a maximum over the grid,
so a duplicate changes nothing, and the budget is inferred from the largest row
sum, which is also unaffected; filtering them out would only hide how the grid
is constructed.


In [ ]:
cf.SyntheticCaseFactory(params).tables()["decision_makers_options"]

## 6. Check the case before trusting it

Before trusting any measurement taken on a case we have to trust the case
itself. `validate_case()` re-imports it and checks a list of properties, raising
an error if any fails:

- every evaluation produces a real number, no NaN or infinity;
- every KPI value stays inside the **envelope**, meaning the lowest and highest
  value that KPI can reach anywhere within the budget. The envelope is worked out
  algebraically beforehand, so this checks the maths against the actual pipeline;
- **no clipping**: tRBS scores a KPI on a 0 to 100 scale between the lowest and
  highest value it observes. If those boundaries sit too close together, values
  get cut off at 0 or 100, which puts artificial creases in the landscape. This
  check confirms the boundaries are wide enough that no cutting off happens;
- the allocation the solver returns respects the budget;
- for convex cases, all starts reached the same score, which is what a single
  mountain implies.

It returns a dictionary of diagnostics rather than a bare pass or fail.

In [ ]:
diagnostics = cf.validate_case(params.name, root, budget=params.budget)

print("Validation diagnostics:")
for key in ["no_clip", "slsqp_consensus", "consensus_spread", "n_distinct_terminal_values"]:
    print(f"  {key}: {diagnostics[key]}")

## 7. The harder cases

Convex cases are the easy baseline. The interesting question is what happens
when the landscape has several peaks, so that where the optimiser starts decides
where it finishes. Two mechanisms produce that, and they are the bottom half of
the catalogue table.

### 7.1 Smooth peaks

The `smooth_nonconvex` regime bends the landscape without creating sharp edges,
through two parameters:

- `n_stb1`: how many KPIs are flipped to smaller-is-better and scored on a curve.
  This is the IZZ cost-KPI mechanism: spending more starts to hurt.
- `n_bilinear`: how many KPIs get a term multiplying two internal variables
  together, so their effects depend on each other rather than simply adding up.

The boundaries stay wide enough that no clipping occurs, so every peak here
comes from genuine curvature and not from an artefact.

In [ ]:
params_smooth = cf.SyntheticCaseParams(
    k=3,
    n_key_outputs=3,
    regime="smooth_nonconvex",
    appreciation="sinusoidal",
    n_stb1=1,
    n_bilinear=1,
    seed=4,
)

root_smooth = cf.SyntheticCaseFactory(params_smooth).write()
print(f"Wrote {params_smooth.name}")

### 7.2 Sharp edges

There is a second route to multiple peaks, and on a small problem it is the one
that actually works. The `nonsmooth` regime deliberately narrows the KPI
boundaries using `bracketing_factor`, a dial between 0 and 1.

At `1.0` the boundaries exactly span what the KPIs can reach, so nothing is cut
off. Below `1.0` they close in, and KPI values start hitting the 0 and 100 ends
of the scale inside the feasible region. Everything past that point scores
identically, which creates flat plateaus with sharp creases at their edges.

This is not a cosmetic difference. Cutting off at the bottom of the scale is
what splits a single peak into two, and it was originally found as a **bug** in
an earlier version of this generator: cases labelled convex were quietly
multi-peaked, so any optimiser measured against them would have been scored
against the wrong answer. It is now kept as a deliberate setting so the effect
can be studied rather than suffered.

The validator recognises the narrowing and confirms it is intentional instead of
raising an error.

In [ ]:
params_sharp = cf.SyntheticCaseParams(
    k=3,
    n_key_outputs=3,
    regime="nonsmooth",
    bracketing_factor=0.6,
    seed=3,
)

root_sharp = cf.SyntheticCaseFactory(params_sharp).write()
diag_sharp = cf.validate_case(params_sharp.name, root_sharp, budget=params_sharp.budget)

print(f"Wrote {params_sharp.name}")
print(f"Clipping-free:           {diag_sharp['no_clip']}   <- False is the point of this case")
print(f"Narrowing margin:        {diag_sharp.get('underbracketing_margin', 'n/a')}")
print(f"Share of points clipped: {diag_sharp.get('clipping_incidence', 'n/a')}")
print()
print("On a convex case a False here would be a defect. Here the narrowing is deliberate,")
print("the validator recognises it as such, and so it reports rather than raises.")

## 8. The standard suite as a regression set

The table in section 3 is not only a menu. It is also the regression set: if a
change to the generator breaks something, it shows up on one of those ten cases.
`cf.standard_cases()` returns them as parameter objects, ready to write.

These ten keep their hand-written names on purpose. They are fixtures referred
to by name in tests and in the write-ups, so a rename would break those
references; everything you generate yourself gets a derived name.

In [ ]:
for p in cf.standard_cases()[:4]:
    print(f"{p.name:32s} regime={p.regime:16s} k={p.k}")
print("...")

## 9. Extending the generator

To add a new regime or mechanism, work through these in order.

1. **`SyntheticCaseParams` and `__post_init__`** in `case_factory.py`: add the
   setting as a field, and add its rules to the validation matrix, for example
   "this setting is only valid in regime X". That matrix is the single source of
   truth for which combinations are allowed. If the setting changes what the case
   *is*, add it to `derived_name()` too, so two cases that differ by it do not
   end up sharing a folder.
2. **`SyntheticCaseFactory`** in `case_factory.py`: draw any new randomness from
   a *new* `SeedSequence` child stream. Never reuse an existing one, because
   that would make an unrelated setting change the draws. Then build the
   mechanism into the relevant table methods, keeping the boundaries wide enough
   that nothing clips, unless clipping is the point of your mechanism.
3. **`envelope()` and `manifest()`** in `case_factory.py`: extend the envelope
   if your mechanism changes which KPI values are reachable, and update the
   convexity claim.
4. **`Oracle.certify`** in `oracle.py`: decide which oracle certifies the new
   regime, and be explicit about whether it proves the answer or estimates it.
5. **`validate_case`** in `case_factory.py`: add the invariant that should hold
   for your regime.
6. **`test_case_factory.py`**: add your case to `test_knob_validation_matrix` so
   invalid combinations are rejected, and add a round-trip test.

### The reproducibility contract

Two tests protect it. `test_determinism_byte_identical` requires that the same
parameters and seed produce byte-identical files. `test_subseed_isolation`
requires that changing one setting does not re-randomise anything unrelated.
Any new mechanism has to keep both passing.

## 10. Next

You now have three cases on disk: one convex, one smoothly bent, one with sharp
edges. None of them has an answer attached yet.

**[Part 2: optimising a synthetic case](02_optimising_a_synthetic_case.ipynb)**
runs them through the optimiser, computes the certified best allocation, and
measures the gap between the two.

Reference:

- [`case_factory.py`](case_factory.py): builds the cases.
- [`oracle.py`](oracle.py): computes the certified answers.
- [`test_case_factory.py`](test_case_factory.py): worked examples and regression
  tests.
- [`vlinder_demo.ipynb`](../../vlinder_demo.ipynb): the basics of tRBS itself.